# Latent-Space Averaging → Neutral HED Scribble

Instead of averaging edge maps in pixel space (which produces blurry nonsense),
we average **VAE latents** of man and woman portraits and decode the result.
The averaged image is then fed into HED to extract a semantically neutral scribble.

**Pipeline:**
1. Generate N man portraits and N woman portraits (sprinter + Sobel oval)
2. VAE-encode each portrait → latent space
3. Average all latents → single "neutral" latent
4. Decode → neutral face image
5. Extract HED scribble from the decoded image
6. Visualize: compare against the original (man-only) scribble baseline
7. Generate sprinter samples from both scribbles to sanity-check neutrality

## 1. Environment Setup

In [ ]:
# Only needed on Colab
import sys
if 'google.colab' in str(get_ipython()):
    !pip install -q diffusers transformers accelerate controlnet_aux
    !pip install -q scikit-learn matplotlib Pillow peft

    from huggingface_hub import login
    from google.colab import userdata
    import getpass

    hf_token = userdata.get('HF') if hasattr(userdata, 'get') else None
    login(token=hf_token) if hf_token else login()

    github_token = userdata.get('GITHUB') if hasattr(userdata, 'get') else getpass.getpass('GitHub token: ')
    import os
    repo_url  = f"https://{github_token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "scribble_cond_loss"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        !cd {repo_name} && git pull
    !cd {repo_name} && git checkout {branch}

    for path in [f'/content/{repo_name}', f'/content/{repo_name}/SD_cond_SD_controlnet']:
        if path not in sys.path:
            sys.path.insert(0, path)
    print(f'✅ Repo ready on branch: {branch}')

## 2. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.decomposition import PCA

from models      import load_models
from image_utils import sobel_proxy, latent_to_pil, build_base_image
from clip_utils  import load_clip_model, encode_images_clip
from generation  import generate_and_store_cs
from visualization import plot_row

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 3. Config

In [ ]:
# ── How many portraits to generate per gender ──────────────────────────
N_PER_GENDER = 5          # increase for a smoother average (at the cost of time)
CONTROLNET_SCALE = 0.5

MAN_PROMPT   = 'a superrealistic portrait photograph of a man, studio lighting'
WOMAN_PROMPT = 'a superrealistic portrait photograph of a woman, studio lighting'
SAMPLE_PROMPT = 'a superrealistic professional photograph of'

# How many sample images to generate from each scribble for comparison
N_SAMPLES = 6

print(f'N_PER_GENDER={N_PER_GENDER}  controlnet_scale={CONTROLNET_SCALE}')

## 4. Load Models

In [ ]:
architect, sprinter = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('✅ Models loaded.')

## 5. Build Oval Base Image & Sobel Conditioning

In [ ]:
base_image_pil, base_tensor = build_base_image(device)

with torch.no_grad():
    sobel_tensor = sobel_proxy(base_tensor, device)
    sobel_pil    = T.ToPILImage()(sobel_tensor.squeeze(0).cpu())

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(base_image_pil); axes[0].set_title('Base Oval');       axes[0].axis('off')
axes[1].imshow(sobel_pil, cmap='gray'); axes[1].set_title('Sobel Cond'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 6. Generate Man & Woman Portraits

In [ ]:
print(f'Generating {N_PER_GENDER} man portraits...')
with torch.no_grad():
    man_images, _ = generate_and_store_cs(
        sprinter, MAN_PROMPT, sobel_pil,
        N_PER_GENDER, batch_size=2, cn_scale=CONTROLNET_SCALE,
    )

print(f'Generating {N_PER_GENDER} woman portraits...')
with torch.no_grad():
    woman_images, _ = generate_and_store_cs(
        sprinter, WOMAN_PROMPT, sobel_pil,
        N_PER_GENDER, batch_size=2, cn_scale=CONTROLNET_SCALE,
    )

plot_row(man_images,   f'Man Portraits (N={N_PER_GENDER})')
plot_row(woman_images, f'Woman Portraits (N={N_PER_GENDER})')

## 7. VAE-Encode All Portraits → Average Latents → Decode

This is the core idea: instead of averaging scribbles (which gives blurry edges),
we average in latent space where interpolation is semantically meaningful.

In [ ]:
def pil_to_latent(pil_img, vae, device):
    """Encode a single PIL image to VAE latent space."""
    tensor = TF.to_tensor(pil_img).unsqueeze(0).to(device).to(torch.float32)
    tensor = (tensor * 2.0) - 1.0  # [0,1] → [-1,1]
    with torch.no_grad():
        latent = vae.encode(tensor).latent_dist.mean
        latent = latent * vae.config.scaling_factor
    return latent  # [1, 4, H/8, W/8]


def decode_latent(latent, vae):
    """Decode a VAE latent to PIL image."""
    with torch.no_grad():
        unscaled = latent / vae.config.scaling_factor
        sample   = vae.decode(unscaled.to(vae.dtype)).sample      # [-1, 1]
        pixel    = torch.clamp((sample.float() + 1.0) / 2.0, 0.0, 1.0)
    return T.ToPILImage()(pixel.squeeze(0).cpu())


# ── Encode all portraits ──────────────────────────────────────────────────────
vae = architect.vae
vae.to(dtype=torch.float32)

all_images = man_images + woman_images
print(f'Encoding {len(all_images)} portraits to VAE latent space...')

latents_list = []
for i, img in enumerate(all_images):
    lat = pil_to_latent(img, vae, device)
    latents_list.append(lat)
    print(f'  [{i+1}/{len(all_images)}] latent shape: {lat.shape}  '
          f'norm: {lat.norm():.2f}')

# ── Stack and average ─────────────────────────────────────────────────────────
latents_stack = torch.cat(latents_list, dim=0)  # [2*N, 4, H/8, W/8]
print(f'\nStacked latents: {latents_stack.shape}')

# Three averages: all, man-only, woman-only — useful for comparison
avg_latent_all   = latents_stack.mean(dim=0, keepdim=True)
avg_latent_man   = torch.cat(latents_list[:N_PER_GENDER]).mean(dim=0, keepdim=True)
avg_latent_woman = torch.cat(latents_list[N_PER_GENDER:]).mean(dim=0, keepdim=True)

print(f'avg_latent_all   norm: {avg_latent_all.norm():.2f}')
print(f'avg_latent_man   norm: {avg_latent_man.norm():.2f}')
print(f'avg_latent_woman norm: {avg_latent_woman.norm():.2f}')

# ── Decode averaged latents ───────────────────────────────────────────────────
decoded_avg_all   = decode_latent(avg_latent_all,   vae)
decoded_avg_man   = decode_latent(avg_latent_man,   vae)
decoded_avg_woman = decode_latent(avg_latent_woman, vae)

# Also decode the middle individual portraits for reference
ref_man   = man_images[N_PER_GENDER // 2]
ref_woman = woman_images[N_PER_GENDER // 2]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img, title in zip(axes,
    [ref_man, ref_woman, decoded_avg_man, decoded_avg_woman, decoded_avg_all],
    ['Ref Man', 'Ref Woman', f'Avg Man (N={N_PER_GENDER})',
     f'Avg Woman (N={N_PER_GENDER})', f'Avg All (N={2*N_PER_GENDER})']):
    ax.imshow(img); ax.set_title(title, fontsize=11); ax.axis('off')
fig.suptitle('Latent-Space Averages vs Reference Portraits', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('✅ Latent averaging complete.')

## 8. Extract HED Scribbles

We extract four scribbles for comparison:
- **Baseline**: from a single man portrait (what `main.ipynb` does with `man_images[2]`)
- **Latent avg all**: from the decoded average of all portraits  ← the main experiment
- **Latent avg man**: from the decoded man average
- **Latent avg woman**: from the decoded woman average

In [ ]:
from controlnet_aux import HEDdetector

print('Loading HED detector...')
hed = HEDdetector.from_pretrained('lllyasviel/Annotators')

def extract_hed(pil_img, label=''):
    scribble = hed(pil_img, scribble=True)
    print(f'  HED extracted{" (" + label + ")" if label else ""}: {scribble.size}')
    return scribble

# Baseline: single man portrait (mirrors main.ipynb line: source_image = man_images[2])
baseline_source   = man_images[min(2, N_PER_GENDER - 1)]
scribble_baseline = extract_hed(baseline_source, 'baseline: single man portrait')

# Latent averages
scribble_avg_all   = extract_hed(decoded_avg_all,   'latent avg all')
scribble_avg_man   = extract_hed(decoded_avg_man,   'latent avg man')
scribble_avg_woman = extract_hed(decoded_avg_woman, 'latent avg woman')

# ── Side-by-side visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Row 0: source images
for ax, img, title in zip(axes[0],
    [baseline_source, decoded_avg_all, decoded_avg_man, decoded_avg_woman],
    ['Source (single man)', 'Decoded avg all', 'Decoded avg man', 'Decoded avg woman']):
    ax.imshow(img); ax.set_title(title, fontsize=11); ax.axis('off')

# Row 1: corresponding HED scribbles
for ax, scr, title in zip(axes[1],
    [scribble_baseline, scribble_avg_all, scribble_avg_man, scribble_avg_woman],
    ['Scribble (baseline)', 'Scribble (avg all) ← use this', 'Scribble (avg man)', 'Scribble (avg woman)']):
    ax.imshow(scr, cmap='gray'); ax.set_title(title, fontsize=11); ax.axis('off')

fig.suptitle('Source Images & Their HED Scribbles', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('✅ All HED scribbles extracted.')

## 9. CLIP Neutrality Check

Measure how far each *scribble* sits from the man and woman CLIP centroids.
A more neutral scribble should have roughly equal distance to both.

In [ ]:
def scribble_to_clip(pil_img, clip_model, clip_processor, device):
    """Encode a single PIL image (e.g. a scribble) to CLIP."""
    tensor = TF.to_tensor(pil_img.convert('RGB')).unsqueeze(0).to(device)
    clip_model.to(device)
    with torch.no_grad():
        emb = encode_images_clip(tensor, clip_model, clip_processor)  # [1, 768]
    clip_model.to('cpu')
    return emb

def portrait_clips(pil_list, clip_model, clip_processor, device):
    """Encode a list of portraits to CLIP; returns [N, 768] tensor."""
    tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in pil_list], dim=0).to(device)
    clip_model.to(device)
    with torch.no_grad():
        embs = encode_images_clip(tensors, clip_model, clip_processor)
    clip_model.to('cpu')
    return embs

# Compute target centroids
man_clips   = portrait_clips(man_images,   clip_model, clip_processor, device)  # [N, 768]
woman_clips = portrait_clips(woman_images, clip_model, clip_processor, device)  # [N, 768]
man_centroid   = man_clips.mean(dim=0)    # [768]
woman_centroid = woman_clips.mean(dim=0)  # [768]

# Encode all four scribble variants
scribbles = {
    'Baseline (single man)': scribble_baseline,
    'Avg all (proposed)':    scribble_avg_all,
    'Avg man':               scribble_avg_man,
    'Avg woman':             scribble_avg_woman,
}

print('CLIP cosine similarity of each scribble to gender centroids:')
print(f'{"Scribble":<28}  sim→man   sim→woman  balance (|Δ|)')
print('-' * 65)

results = {}
for name, scr in scribbles.items():
    emb = scribble_to_clip(scr, clip_model, clip_processor, device).squeeze(0)  # [768]
    sim_man   = (emb @ man_centroid).item()   / (emb.norm() * man_centroid.norm()).item()
    sim_woman = (emb @ woman_centroid).item() / (emb.norm() * woman_centroid.norm()).item()
    balance   = abs(sim_man - sim_woman)
    results[name] = (sim_man, sim_woman, balance)
    print(f'{name:<28}  {sim_man:.4f}   {sim_woman:.4f}   {balance:.4f}')

best = min(results, key=lambda k: results[k][2])
print(f'\n✅ Most gender-neutral scribble: "{best}"  (|Δ|={results[best][2]:.4f})')

## 10. PCA Visualisation — Where Do the Scribbles Land?

In [ ]:
# Gather all embeddings for PCA
scribble_embs = {}
for name, scr in scribbles.items():
    emb = scribble_to_clip(scr, clip_model, clip_processor, device)
    scribble_embs[name] = emb.cpu().numpy()  # [1, 768]

all_target = torch.cat([man_clips, woman_clips], dim=0).cpu().numpy()  # [2N, 768]
scr_stack  = np.vstack(list(scribble_embs.values()))                   # [4, 768]

combined = np.vstack([all_target, scr_stack])
pca      = PCA(n_components=2)
coords   = pca.fit_transform(combined)

n_target = all_target.shape[0]
target_coords  = coords[:n_target]
scribble_coords = coords[n_target:]

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(target_coords[:N_PER_GENDER, 0], target_coords[:N_PER_GENDER, 1],
           c='royalblue', alpha=0.6, s=60, label='Target (man)')
ax.scatter(target_coords[N_PER_GENDER:, 0], target_coords[N_PER_GENDER:, 1],
           c='crimson', alpha=0.6, s=60, label='Target (woman)')

markers    = ['D', 'x', '^', 's']
scr_colors = ['black', 'limegreen', 'steelblue', 'tomato']
for (name, _), coord, marker, color in zip(scribbles.items(), scribble_coords, markers, scr_colors):
    ax.scatter(coord[0], coord[1], c=color, s=120, marker=marker,
               zorder=5, label=f'Scribble: {name}')

ax.set_title(f'CLIP PCA — Targets vs Scribble Embeddings\n'
             f'Variance explained: {pca.explained_variance_ratio_.sum():.1%}',
             fontsize=13)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Sprinter Samples from Each Scribble

The real test: what does the sprinter generate from each scribble?
The latent-average scribble should produce a mix of genders rather than predominantly male.

In [ ]:
def generate_from_scribble(scribble_pil, prompt, n, controlnet_scale=0.5):
    """Generate N portraits from a scribble using the sprinter."""
    original_vae_dtype = sprinter.vae.dtype
    sprinter.vae.to(dtype=torch.float16)
    images = []
    with torch.no_grad():
        for start in range(0, n, 2):
            bs = min(2, n - start)
            result = sprinter(
                prompt=[prompt] * bs,
                image=[scribble_pil] * bs,
                num_inference_steps=2,
                guidance_scale=0.0,
                controlnet_conditioning_scale=controlnet_scale,
                output_type='pil',
            )
            images.extend(result.images)
    sprinter.vae.to(dtype=original_vae_dtype)
    return images


print(f'Generating {N_SAMPLES} sprinter samples from each scribble...')

samples = {}
for name, scr in scribbles.items():
    print(f'  Generating from: {name}')
    imgs = generate_from_scribble(scr, SAMPLE_PROMPT, N_SAMPLES, CONTROLNET_SCALE)
    samples[name] = imgs

# Plot all
n_rows = len(scribbles)
fig, axes = plt.subplots(n_rows, N_SAMPLES + 1, figsize=(3 * (N_SAMPLES + 1), 3.5 * n_rows))

for row_idx, (name, scr) in enumerate(scribbles.items()):
    # First column: the scribble itself
    axes[row_idx, 0].imshow(scr, cmap='gray')
    axes[row_idx, 0].set_title('Scribble', fontsize=9)
    axes[row_idx, 0].set_ylabel(name, fontsize=9, rotation=0, labelpad=100, va='center')
    axes[row_idx, 0].axis('off')
    # Remaining columns: sprinter samples
    for col_idx, img in enumerate(samples[name]):
        axes[row_idx, col_idx + 1].imshow(img)
        axes[row_idx, col_idx + 1].axis('off')

fig.suptitle(f'Sprinter Samples from Each Scribble (N={N_SAMPLES} per row)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Done.')

## 12. Quick CLIP Gender Classification on Sprinter Samples

Zero-shot softmax over 'man' vs 'woman' text prompts for each sample set.
A neutral scribble should produce roughly 50/50.

In [ ]:
import torch.nn.functional as F

text_inputs = clip_processor(
    text=[MAN_PROMPT, WOMAN_PROMPT],
    return_tensors='pt', padding=True,
).to(device)

clip_model.to(device)
with torch.no_grad():
    text_feats = clip_model.get_text_features(**text_inputs)
    text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

print(f'{"Scribble":<28}  %male  %female  label')
print('-' * 55)

for name, imgs in samples.items():
    tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in imgs], dim=0).to(device)
    with torch.no_grad():
        img_feats = encode_images_clip(tensors, clip_model, clip_processor)  # [N, 768]
    logits = img_feats @ text_feats.T * 100.0   # [N, 2]
    probs  = F.softmax(logits, dim=-1).cpu().numpy()
    p_male   = probs[:, 0].mean()
    p_female = probs[:, 1].mean()
    label    = 'male' if p_male > 0.5 else 'female'
    print(f'{name:<28}  {p_male:.1%}  {p_female:.1%}   {label}')

clip_model.to('cpu')
print('\n✅ Gender classification complete.')

## 13. Save the Neutral Scribble for Use in `run_dps.py`

In [ ]:
import os

out_dir = 'output/neutral_scribble'
os.makedirs(out_dir, exist_ok=True)

# Save all variants
decoded_avg_all.save(f'{out_dir}/decoded_avg_all.png')
decoded_avg_man.save(f'{out_dir}/decoded_avg_man.png')
decoded_avg_woman.save(f'{out_dir}/decoded_avg_woman.png')

scribble_baseline.save(f'{out_dir}/scribble_baseline_man.png')
scribble_avg_all.save(f'{out_dir}/scribble_avg_all.png')       # ← use this in run_dps.py
scribble_avg_man.save(f'{out_dir}/scribble_avg_man.png')
scribble_avg_woman.save(f'{out_dir}/scribble_avg_woman.png')

print(f'✅ Saved to {out_dir}/')
print()
print('To use the neutral scribble in run_dps.py, modify extract_scribble_hed() to')
print('load scribble_avg_all.png instead of extracting from a single man portrait:')
print()
print('  from PIL import Image')
print(f'  scribble_pil = Image.open("{out_dir}/scribble_avg_all.png")')

## 14. (Optional) Ablation: How Many Portraits Needed?

Average latents from 1, 2, 3, ... N portraits per gender and see when the decoded
image stabilises. Helps choose a good `N_PER_GENDER`.

In [ ]:
# Only run if you have more than 1 image per gender
if N_PER_GENDER > 1:
    ns = list(range(1, N_PER_GENDER + 1))
    decoded_by_n = []
    for n in ns:
        lat = torch.cat(latents_list[:n] + latents_list[N_PER_GENDER:N_PER_GENDER + n]).mean(dim=0, keepdim=True)
        decoded_by_n.append(decode_latent(lat, vae))

    fig, axes = plt.subplots(1, len(ns), figsize=(4 * len(ns), 4))
    if len(ns) == 1:
        axes = [axes]
    for ax, img, n in zip(axes, decoded_by_n, ns):
        ax.imshow(img); ax.set_title(f'N={n} per gender'); ax.axis('off')
    fig.suptitle('Decoded Latent Average — N Ablation', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Set N_PER_GENDER > 1 to run this ablation.')